In [1]:
import ast
import os

In [2]:
from urllib.parse import urlparse

def get_filename(url: str):
    if url[-4:] == ".git":
        url = url[:-4]
        
    parts = urlparse(url).path.strip("/").split("/")

    if len(parts) >= 2:
        username = parts[0]
        project_name = parts[1]
        result = f"{username}-{project_name}"
        return result

In [3]:
repo = "https://github.com/princ3kr/Notebook-LM-Mini"

filename = get_filename(repo)
dir = f"../temp/{filename}"

if os.path.isdir(dir):
    print("Directory already exists")

Directory already exists


In [4]:
import subprocess

if os.path.isdir(dir):
    print("Directory already exists")
else:
    try:
        subprocess.run(["git", "clone", repo, dir], check=True)
        print("Clone successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed with error: {e}")


Directory already exists


In [5]:
# tree = ast.parse(files["src\\services\\tutor_service.py"])
# print(ast.dump(tree, indent=2))

def parse_file(source_code):
    tree = ast.parse(source_code)
    import_modules, import_names, classes, functions = [], [], [], []
    
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            import_modules.append(node.module)
            import_names.append([alias.name for alias in node.names])

        if isinstance(node, ast.ClassDef):
            classes.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
            
        if isinstance(node, ast.FunctionDef):
            functions.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
                
    return {"import_modules": import_modules, "import_names": import_names, "classes": classes, "functions": functions}

In [6]:
ignores = { ".git", ".gitignore", ".lock", ".venv", "__pycache__", "node_modules", ".vscode", "pyproject.toml", "__init__.py", ".python-version", "requirements.txt" }

def get_structure(directory: str):
    for root, dirs, files in os.walk(directory):
        dirs[:] = sorted([d for d in dirs if d not in ignores])

        rel_dir = os.path.relpath(root, directory)
        
        if rel_dir == ".":
            level = 0
            display_name = os.path.basename(os.path.abspath(directory))
        else:
            level = rel_dir.count(os.sep) + 1
            display_name = os.path.basename(root)

        indent = "│   " * (level - 1) if level > 0 else ""
        connector = "├── " if level > 0 else ""
        print(f"{indent}{connector}{display_name}/")

        file_indent = "│   " * level
        sorted_files = sorted(files)
        for i, name in enumerate(sorted_files):
            if name not in ignores or name == "README.md":
                file_connector = "└── " if (i == len(sorted_files) - 1 and not dirs) else "├── "
                print(f"{file_indent}{file_connector}{name}")


def get_files(directory: str):
    inventory = {}
    for root, dirs, files in os.walk(directory):
        rel_dir = os.path.relpath(root, directory)
        if rel_dir == ".":
            rel_dir = ""
        
        for name in files:
            if (name not in ignores) and (name == "README.md" or name.endswith(".py")):
                full_path = os.path.join(root, name)

                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()

                    parsed_structure = {"import_modules": [], "import_names": [], "classes": [], "functions": []}

                    if name.endswith(".py"):
                        parsed_structure = parse_file(content)

                    parsed_structure['content'] = content

                    inventory[os.path.join(rel_dir, name)] = parsed_structure
                        

    return inventory

In [7]:
# get_structure(dir)
files = get_files(directory=dir)

In [8]:
for imports in files['src\services\diagnoser_service.py']:
    print(imports)

import_modules
import_names
classes
functions
content


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_13920\1899480365.py:1: SyntaxWarning: invalid escape sequence '\s'
  for imports in files['src\services\diagnoser_service.py']:


In [9]:
import networkx as nx
from pyvis.network import Network
from neo4j import GraphDatabase
import json

class BuildGraph:
    def __init__(self, files: dict):
        self.G = nx.DiGraph()
        self.files = {filepath.replace("\\", "/"): data for filepath, data in files.items()}
        self.name_index = self.build_name_index()

    def build_name_index(self) -> dict:
        name_index = {}
        for filepath in self.files:
            for classname in self.files.get(filepath, {}).get("classes", []):
                name_index[classname['name']] = filepath
            for funcname in self.files.get(filepath, {}).get("functions", []):
                name_index[funcname['name']] = filepath
        return name_index

    def get_path(self) -> dict:
        cleared_files = {}
        for filepath in self.files:
            imports = []
            modules = self.files[filepath].get("import_modules", [])
            names = self.files[filepath].get("import_names", [])

            for imp, imp_names in zip(modules, names):
                if not imp or not imp.startswith("src."): ## harcoded!
                    continue

                target = imp.replace(".", "/") + ".py" ## hardcoded!
                if target in self.files:
                    imports.append(target)
                else:
                    for name in imp_names:
                        if name in self.name_index:
                            imports.append(self.name_index[name])
                            
            cleared_files[filepath] = imports
        return cleared_files

    def get_node_style(self, filepath):
        """Returns a more premium color palette and node properties."""
        if "services" in filepath:
            color = {"background": "#3b82f6", "border": "#60a5fa", "highlight": "#93c5fd"}
        elif "models" in filepath:
            color = {"background": "#10b981", "border": "#34d399", "highlight": "#6ee7b7"}
        elif "database" in filepath:
            color = {"background": "#f59e0b", "border": "#fbbf24", "highlight": "#fcd34d"}
        else:
            color = {"background": "#8b5cf6", "border": "#a78bfa", "highlight": "#c4b5fd"}
        
        return color

    def build(self):
        cleared_path = self.get_path()
        for source, targets in cleared_path.items():
            style = self.get_node_style(source)
            self.G.add_node(
                source, 
                label=source.split("/")[-1],
                title=source,
                color=style,
                shadow=True,
                shape="dot",
                size=25,
                font={"color": "white", "size": 14, "face": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}
            )
            for target in targets:
                self.G.add_edge(source, target)

    def push_to_neo4j(self, uri, user, password):
        driver = GraphDatabase.driver(
            uri, auth=(user, password)
        )
        
        repo_id = filename

        with driver.session() as session:
            for node in self.G.nodes():
                imports = [
                    name for names in self.files[node]['import_names'] 
                    for name in names if name in self.name_index
                ]

                query = '''
                    MERGE (r:Repo {repo_id: $repo_id})
                    MERGE (f:File {path: $path, repo_id: $repo_id})
                    MERGE (r)-[:CONTAINS]->(f)
                    SET f.name = $name,
                        f.classes = $classes,
                        f.imports = $imports,
                        f.functions = $functions
                '''
                session.run(
                    query,
                    name=node.split('/')[-1],
                    path=node,
                    classes=[c['name'] for c in self.files.get(node, {}).get("classes", [])],
                    imports=imports,
                    functions=[f['name'] for f in self.files.get(node, {}).get("functions", [])],
                    repo_id=repo_id
                )
                
            for source, target in self.G.edges():
                query = '''
                    MATCH (a:File {path: $source, repo_id: $repo_id})
                    MATCH (b:File {path: $target, repo_id: $repo_id})
                    MERGE (a)-[:IMPORTS]->(b)
                '''
                session.run(query, source=source, target=target, repo_id=repo_id)
                

    def show(self, filename="graph.html"):
        net = Network(
            directed=True, 
            notebook=True, 
            cdn_resources='remote', 
            height="750px", 
            width="100%", 
            bgcolor="#111827",
            font_color="white"
        )
        
        net.from_nx(self.G)

        options = {
            "edges": {
                "color": {"inherit": "from", "opacity": 0.5},
                "smooth": {"type": "continuous", "roundness": 0.5},
                "arrows": {"to": {"enabled": True, "scaleFactor": 0.5}},
                "width": 1.5
            },
            "physics": {
                "forceAtlas2Based": {
                    "gravitationalConstant": -60,
                    "centralGravity": 0.005,
                    "springLength": 150,
                    "springStrength": 0.08,
                    "damping": 0.4,
                    "avoidOverlap": 0.5
                },
                "maxVelocity": 50,
                "minVelocity": 0.1,
                "solver": "forceAtlas2Based",
                "stabilization": {"enabled": True, "iterations": 1000}
            },
            "interaction": {
                "hover": True,
                "navigationButtons": True,
                "multiselect": True,
                "tooltipDelay": 200
            }
        }
        
        net.set_options(json.dumps(options))
        
        return net.write_html(filename)


In [10]:
builder = BuildGraph(files)
builder.build()
builder.show()

In [34]:
builder.push_to_neo4j("bolt://localhost:7687", "neo4j", "pk142145")

In [11]:
import chromadb
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')

class VectorStore:
    def __init__(self, files:dict, path='../chromadb', collection_name:str='database'):
        self.files = files
        self.client = chromadb.PersistentClient(path=path)
        ef = OpenAIEmbeddingFunction(api_key=openai_key, model_name="text-embedding-3-small")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=ef)

    def build(self, max_module_lines=100, overlap=10):
        self.chunks = []
        for file in self.files.keys():
            content = self.files[file]['content']
            lines = content.split('\n')
            classes = self.files[file]['classes']
            functions = self.files[file]['functions']

            for i in range(0, len(lines), max_module_lines):
                start = max(0, i - overlap)
                end = min(len(lines), i + overlap + max_module_lines)
                chunk_lines = lines[start : end]
                self.chunks.append([
                    "\n".join(chunk_lines),
                    {
                        "path": file,
                        "filename": file.split("/")[-1],
                        "function_name": "module_level_segment",
                        "class_name": "module_level",
                        "line_start": start + 1,
                        "line_end": end
                    }
                ])

            for func in functions:
                parent = 'module_level'
                for c in classes:
                    if func['line_start'] >= c['line_start'] and func['line_start'] <= c['line_end']:
                        parent = c['name']
                        break
                
                start = max(0, func['line_start'] - overlap)
                end = min(len(lines), func['line_end'] + overlap)
                chunk = "\n".join(lines[start:end])
                metadata = {
                    "path": file.replace(),
                    "filename": file.split("/")[-1],
                    "function_name": func['name'],
                    "class_name": parent,
                    "line_start": start + 1,
                    "line_end": end
                }
                self.chunks.append([chunk, metadata])
                self.push()

    def push(self):
        if not self.chunks:
            print("No chunks to push. Run build() first.")
            return

        documents = [c[0] for c in self.chunks]
        metadatas = [c[1] for c in self.chunks]

        ids = [f"{m['path']}:{m['function_name']}:{i}" for i, m in enumerate(metadatas)]

        self.collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas
        )
        print(f"Successfully pushed {len(documents)} chunks to collection '{self.collection.name}'.")


    def search(self, query: str, top_k: int = 5):
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        return results

In [12]:
client = VectorStore(files, collection_name='prototype')
# client.build()

In [32]:
# client.push()

In [13]:
results = client.search(
    query="what is the function tutor is doing and how many files it is importing from"
)
# (results['ids'])


In [14]:
results

{'ids': [['src\\services\\tutor_service.py:__init__:76',
   'app.py:module_level_segment:1',
   'app.py:module_level_segment:0',
   'src\\services\\tutor_service.py:module_level_segment:71',
   'README.md:module_level_segment:3']],
 'embeddings': None,
 'documents': [['from src.services.tutor_lesson_utils import (\n    PROCEED_PROMPT,\n    format_equations_for_prompt,\n    is_new_topic_intent,\n    is_proceed_to_quiz,\n)\n\n\nclass TutorWorkflow:\n    def __init__(self, neo4j_conn: Neo4jConn):\n        self.neo4j_conn = neo4j_conn\n        self.workflow = self._build_workflow()\n        self.app = self.workflow.compile(checkpointer=MemorySaver())\n        self.llm = ChatGroq(\n            model="llama-3.3-70b-versatile", \n            temperature=0.3, \n            max_tokens=250\n        )\n        # Longer explanations for the teach-first phase\n        self.teach_llm = ChatGroq(\n            model="llama-3.3-70b-versatile",\n            temperature=0.35,\n            max_tokens=1000

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal
from sentence_transformers import CrossEncoder
from langchain_core.messages import SystemMessage, HumanMessage

groq_api_key = os.getenv('GROQ_API_KEY')

class CypherQuery(BaseModel):
    cypher: str = Field(..., description="Cypher query for retrieving relationships within the nodes")

class RouterDecision(BaseModel):
    decision: Literal['hybrid', 'graph'] = Field(..., description="Graph for only relationships between nodes/files and hybrid for both graph and vector similarity search")
    reasoning: str = Field(..., description="Brief reasoning behind the routing decision")
    confidence_score: float = Field(..., description="Confidence score of the decision")

class QueryEngine:
    def __init__(self, repo_id: str, db_client: VectorStore, uri:str, user:str, password:str, llm: ChatGroq):
        self.db_client = db_client
        self.repo_id = repo_id
        self.llm = llm
        self.graph_driver = GraphDatabase.driver(uri, auth=(user, password))
        self.rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def vector_search(self, query: str, filenames: list[str]):
        collection = self.db_client.collection
        
        return collection.query(
            query_texts=[query],
            n_results=10,
            where={"path": {"$in": filenames}}
        )

    def graph_search(self, query):
        structured_llm = self.llm.with_structured_output(CypherQuery)

        system_content = f"""
        You are a Cypher query expert for Neo4j. Generate ONLY valid Cypher queries.
        
        SCHEMA:
        - Repo node with property repo_id
        - (Repo)-[:CONTAINS]->(File)
        - (File)-[:IMPORTS]->(File)
        
        FILE PROPERTIES (the ONLY valid properties):
        - name: string (just filename e.g. app.py)
        - path: string (full path e.g. src/services/tutor_service.py)
        - functions: list of strings
        - classes: list of strings
        - imports: list of strings
        
        CRITICAL RULES:
        1. Always start with: MATCH (r:Repo {{repo_id: '{self.repo_id}'}})
        2. ONLY return f.path and imported.path
        3. Use OPTIONAL MATCH for relationships that may not exist
        4. NEVER use: Type(), labels(), IS EMPTY, IS NOT EMPTY
        5. NEVER invent properties — only use name, path, functions, classes, imports
        6. For list contents use: 'value' IN f.functions or 'value' IN f.classes
        7. Only valid WHERE operators: =, <>, IN, CONTAINS, STARTS WITH, IS NULL, IS NOT NULL
        8. For indirect dependencies use: [:IMPORTS*1..2]
        
        EXAMPLES:
        
        Get file and its imports:
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File {{name: 'app.py'}})
        OPTIONAL MATCH (f)-[:IMPORTS]->(imported)
        RETURN f.path, imported.path
        
        Find files with no imports:
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
        WHERE NOT (f)-[:IMPORTS]->()
        RETURN f.path, null as imported_path
        
        Indirect dependencies:
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File {{name: 'app.py'}})
        OPTIONAL MATCH (f)-[:IMPORTS*1..2]->(imported)
        RETURN f.path, imported.path
        """

        response = structured_llm.invoke([
            SystemMessage(content=system_content),
            HumanMessage(content=f"Question: {query}")
        ])

        try:
            with self.graph_driver.session() as session:
                result = session.run(response.cypher)
                output = [record.data() for record in result]
            return output
        except Exception as e:
            print(f"Cypher error: {e}")
            print(f"Failed query: {response.cypher}")
            return []
    
    def rerank(self, context: dict, query: str, top_k: int):
        documents = context['documents'][0]
        metadatas = context['metadatas'][0]

        chunks = [(query, chunk) for chunk in documents]

        scores = self.rerank_model.predict(chunks)
        
        result = []

        for i, score in enumerate(scores):
            result.append([metadatas[i], score, documents[i]])

        result = sorted(result, key=lambda x: x[1], reverse=True)[:top_k]

        return result

    def router(self, query: str) -> RouterDecision:
        router_llm = self.llm.with_structured_output(RouterDecision)
        router_prompt = ChatPromptTemplate.from_messages([
            ('system', """
                You are a query router for a code repository assistant.
                Classify the query into one of three types:

                - graph: query asks about file relationships, imports, dependencies, structure
                - hybrid: query asks about all, structure, logic, function behavior, implementation detailsand implementation

                Return your decision as structured output.
            """),
            ('user', "query: {query}")
        ])

        router_chain = router_prompt | router_llm

        response = router_chain.invoke({"query": query})

        return response
    
    def get_result(self, query: str, top_k: int):
        decision = self.router(query)

        route = decision.decision
        reasoning = decision.reasoning
        score = decision.confidence_score

        if route == "graph":
            graph_data = self.graph_search(query)

            if not graph_data:
                vector_data = self.db_client.collection.query(
                    query_texts=[query],
                    n_results=top_k
                )

                final_vector_data = self.rerank(vector_data, query, top_k)
                return {
                    "type": "hybrid",
                    "graph": [],
                    "vector": final_vector_data,
                    "confidence": score,
                    "reasoning": reasoning
                }

            return {
                "type": "graph",
                "graph": graph_data,
                "confidence": score,
                "reasoning": reasoning
            }

        else:
            graph_data = self.graph_search(query)
            
            filenames = list(set([
                v for r in graph_data
                for v in r.values()
                if v is not None and isinstance(v, str)
            ]))

            if not filenames:
                vector_data = self.db_client.collection.query(
                    query_texts=[query],
                    n_results=top_k
                )

                final_vector_data = self.rerank(vector_data, query, top_k)

                return {
                    "type": "hybrid",
                    "graph": graph_data,
                    "vector": final_vector_data,
                    "confidence": score,
                    "reasoning": reasoning
                }
            
            else:
                vector_data = self.vector_search(query, filenames=filenames)

                final_vector_data = self.rerank(vector_data, query, top_k)
            
                return {
                    "type": "hybrid",
                    "graph": graph_data,
                    "vector": final_vector_data,
                    "confidence": score,
                    "reasoning": reasoning
                }
            
    def close(self):
        self.graph_driver.close()
    

In [75]:
from langchain_openai import ChatOpenAI
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    max_tokens=200,
    api_key=groq_api_key
)
openai_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

engine = QueryEngine(repo_id=filename, db_client=client, uri="bolt://localhost:7687", user="neo4j", password="pk142145", llm=openai_llm)
query = "How the _render_teach_lesson function is working?"
result = engine.get_result(query, 6)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10387.33it/s]


In [76]:
result.keys()

dict_keys(['type', 'graph', 'vector', 'confidence', 'reasoning'])

In [77]:
result['vector']

[[{'path': 'src\\services\\tutor_service.py',
   'function_name': '_render_teach_lesson',
   'line_start': 109,
   'filename': 'src\\services\\tutor_service.py',
   'line_end': 179,
   'class_name': 'TutorWorkflow'},
  np.float32(0.7204324),
  '            goto="human",\n            update={\n                "phase": "teach",\n                "final_response": final_response,\n                "current_question": PROCEED_PROMPT,\n                "student_answer": "",\n            },\n        )\n\n    def _render_teach_lesson(self, concept_name: str) -> str:\n        """Build the lesson text (including equations if present)."""\n        diagnoser = DiagnoserService(self.neo4j_conn)\n        meta = diagnoser.fetch_concept_metadata(concept_name)\n        equations_block, has_equations = format_equations_for_prompt(meta.get("equations"))\n\n        equation_rules = ""\n        if has_equations:\n            equation_rules = """s\n            - The knowledge base includes **equations** for t

In [78]:
def formate_documents(data) -> str:
    context = ""

    if data['type'] in ('graph', 'hybrid'):
        graph_data = data['graph']
        context += f"Relationships:\n"
        for r in graph_data:
            context += f" {r.get('f.path', '')} -> {r.get('imported.path', '')} \n"
        
    if data['type'] == 'hybrid':
        context += "CHUNK DATA:\n"

        for vector_data in data['vector']:
            docs = vector_data[2]
            metas = vector_data[0]
            context += f"\n[{metas['path']} | lines {metas['line_start']}-{metas['line_end']}]\n{docs}\n"

    return context
        

In [79]:
formated_result = formate_documents(result)

In [80]:
from pydantic import BaseModel, Field

class ResponseModel(BaseModel):
    answer: str = Field(..., description="Response of the query from the llm model")
    score: float = Field(..., description="Confidence score of the response")
    sources: str = Field(..., description="File name used for response")

class AnswerEngine:
    def __init__(self, llm):
        self.llm = llm.with_structured_output(ResponseModel)

    def generate_response(self, query:str, context: str):
        prompt = ChatPromptTemplate.from_messages([
            ("system", """
                You are a code repository assistant.
                Answer only from the provided context.
                Sources should be file names used to answer.
                Confidence should be between 0 and 1.
            """), ("user", "Context:\n{context}\n\nQuestion: {query}")
        ])

        chain = prompt | self.llm

        response = chain.invoke({"context": context, "query": query})

        return response


In [81]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# query = "what if i delete concept.py"
chat_engine = AnswerEngine(openai_llm)
result = chat_engine.generate_response(query, formated_result)

In [82]:
result.answer

"The `_render_teach_lesson` function is responsible for generating a lesson text for a given concept. Here's how it works:\n\n1. **Fetch Metadata**: It uses the `DiagnoserService` to fetch metadata for the specified `concept_name`. This metadata includes information such as equations, chunk type, difficulty score, description, and subtopics.\n\n2. **Format Equations**: If the concept includes equations, it formats them for inclusion in the lesson using the `format_equations_for_prompt` function. It also sets rules for how these equations should be presented in the lesson.\n\n3. **Build System Text**: It constructs a `system_text` string that outlines the rules for teaching the concept. This includes providing a clear lesson with an overview, definitions, key relationships, and examples, while also specifying how to handle equations if they are present.\n\n4. **Build Human Text**: It constructs a `human_text` string that includes the concept name, chunk type, difficulty, description, an

## RAG Evaluation metrices

In [83]:
eval_questions = [

    # --- EDGE: FILES WITH NO IMPORTS ---
    {
        "question": "Which files in the repository do not import any other file?",
        "ground_truth": "Files with no imports are leaf nodes in the dependency graph — likely src/models/concept.py, src/models/state.py, src/models/diagnoses.py, and src/models/intent.py as they are pure model definitions"
    },

    # --- EDGE: CIRCULAR / DEEP DEPENDENCY ---
    {
        "question": "Which service has the most dependencies on other files?",
        "ground_truth": "src/services/tutor_service.py has the most dependencies as it imports Neo4jConn, StudentDB, DiagnoserService, IntentService, PlannerService, and tutor_lesson_utils"
    },

    # --- EDGE: SPECIFIC FUNCTION BEHAVIOR ---
    {
        "question": "What happens if planned_paths is empty in the planner node?",
        "ground_truth": "If planned_paths is empty, the planner node returns Command with goto=END and a final_response message explaining either the topic was not recognized or no learning path was found"
    },

    # --- EDGE: SPECIFIC CLASS METHOD ---
    {
        "question": "What does update_last_json_path do in StudentDB and when is it called?",
        "ground_truth": "update_last_json_path updates the last_json_path field and last_updated timestamp in TinyDB for the student. It is called in app.py after successfully building the knowledge graph from a PDF"
    },

    # --- EDGE: STATE FIELD USAGE ---
    {
        "question": "What is the initial value of phase in GraphState when a new session starts?",
        "ground_truth": "The initial value of phase is 'quiz' as set in the initial_state construction in app.py"
    },

    # --- EDGE: ERROR PATH ---
    {
        "question": "What does app.py do if no concepts are extracted from the uploaded PDF?",
        "ground_truth": "app.py updates status to error state, displays an error message saying no relevant concepts were identified and to try a more detailed document, then calls st.stop()"
    },

    # --- EDGE: MULTI-HOP DEPENDENCY ---
    {
        "question": "Which files does app.py indirectly depend on through tutor_service.py?",
        "ground_truth": "Through tutor_service.py, app.py indirectly depends on neo4j_conn.py, student_db.py, diagnoser_service.py, intent_service.py, planner_service.py, and tutor_lesson_utils.py"
    },

    # --- EDGE: SPECIFIC PARAMETER ---
    {
        "question": "What is the process_limit in app.py and what does it control?",
        "ground_truth": "process_limit is set to 50 in app.py and controls the maximum number of PDF chunks processed for concept extraction to avoid excessive API calls"
    },

    # --- EDGE: TWO LLM INSTANCES ---
    {
        "question": "Why does TutorWorkflow initialize two separate LLM instances?",
        "ground_truth": "TutorWorkflow uses self.llm with max_tokens=250 for quick responses like quiz questions and self.teach_llm with max_tokens=1000 and temperature=0.35 for longer concept explanations requiring more detail"
    },

    # --- EDGE: CHECKPOINT ---
    {
        "question": "What checkpointing mechanism does TutorWorkflow use and why?",
        "ground_truth": "TutorWorkflow uses MemorySaver from langgraph.checkpoint.memory as the checkpointer when compiling the workflow, which enables conversation state persistence across multiple turns using thread_id"
    },

    # --- EDGE: THREAD ID ---
    {
        "question": "How is thread_id constructed in app.py and what is it used for?",
        "ground_truth": "thread_id is constructed as f'thread_{student_id}' in app.py and is used as the LangGraph config key to isolate each student's conversation state in MemorySaver"
    },

    # --- EDGE: RESUME VS FRESH ---
    {
        "question": "What condition determines whether the tutor resumes an existing session or starts fresh?",
        "ground_truth": "If state.values and state.next both exist, the tutor resumes via Command(resume=user_input). If either is falsy, it starts fresh by invoking with a new GraphState"
    },

    # --- EDGE: TEMP FILE CLEANUP ---
    {
        "question": "How does app.py handle the temporary PDF file after processing?",
        "ground_truth": "After processing, app.py checks if the temp file exists using os.path.exists and removes it using os.remove to clean up disk space"
    },

    # --- EDGE: MISSING INPUT GUARD ---
    {
        "question": "How does app.py handle the case where user types the placeholder text 'I want to learn about...' literally?",
        "ground_truth": "app.py checks if user_input.strip().lower() equals 'i want to learn about...' or 'i want to learn about' and replaces user_input with an empty string to avoid sending the placeholder as actual input"
    },

    # --- EDGE: DATABASE LAYER ---
    {
        "question": "What database does StudentDB use internally and what file does it store data in?",
        "ground_truth": "StudentDB uses TinyDB internally and stores data in a JSON file at the path db_folder/student_sessions.json where db_folder is determined by the student_id"
    }
]

In [84]:
results = []

for sample in eval_questions:
    raw_result = engine.get_result(sample["question"], top_k=5)
    
    context = formate_documents(raw_result)
    
    response = chat_engine.generate_response(sample["question"], context)
    
    chunks = []

    if raw_result["type"] in ("graph", "hybrid"):
        chunks.append(str(raw_result["graph"]))

    if raw_result["type"] == "hybrid":
        chunks.extend(data[2] for data in raw_result["vector"])

    results.append({
        "question": sample["question"],
        "answer": response.answer,
        "contexts": chunks,
        "ground_truth": sample["ground_truth"]
    })

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `planned_paths` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=9, offset=84>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 84, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (r:Repo {repo_id: 'princ3kr-Notebook-LM-Mini'})-[:CONTAINS]->(f:File)\nWHERE f.planned_paths IS NULL OR f.planned_paths = []\nRETURN f.path, null as imported_path"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The prope

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `planned_paths` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=9, offset=84>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 84, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (r:Repo {repo_id: 'princ3kr-Notebook-LM-Mini'})-[:CONTAINS]->(f:File)\nWHERE f.planned_paths IS NULL OR f.planned_paths = []\nRETURN f.path, null as imported_path"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The prope

TypeError: unhashable type: 'list'

In [92]:
results

[{'question': 'Which files in the repository do not import any other file?',
  'answer': 'The files that do not import any other file are:\n\n- `src/models/diagnoses.py`\n- `src/models/intent.py`\n- `src/models/concept.py`\n- `src/models/state.py`\n- `src/services/pdf_service.py`\n- `README.md`',
  'contexts': ["[{'f.path': 'src/models/diagnoses.py'}, {'f.path': 'src/models/intent.py'}, {'f.path': 'src/services/tutor_lesson_utils.py'}, {'f.path': 'src/models/concept.py'}, {'f.path': 'src/models/state.py'}, {'f.path': 'src/services/pdf_service.py'}, {'f.path': 'src/database/student_db.py'}, {'f.path': 'src/database/neo4j_conn.py'}, {'f.path': 'README.md'}]"],
  'ground_truth': 'Files with no imports are leaf nodes in the dependency graph — likely src/models/concept.py, src/models/state.py, src/models/diagnoses.py, and src/models/intent.py as they are pure model definitions'},
 {'question': 'Which service has the most dependencies on other files?',
  'answer': 'The context provided does 

In [93]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini"
))

dataset = Dataset.from_list(results)

scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm
)

print(scores)

Evaluating: 100%|██████████| 75/75 [00:44<00:00,  1.70it/s]


{'faithfulness': 0.6229, 'answer_relevancy': 0.7619, 'answer_correctness': 0.5962, 'context_precision': 0.4681, 'context_recall': 0.5556}


In [39]:
from typing import TypedDict

class QueryState(TypedDict):
    question: str
    vector_context: str
    graph_context: str
    agent: str
    answer: str
